# Update 1 · Evolución del notebook 04

**Base:** `04_Rendimiento_y_eventos.ipynb`, conservado sin modificaciones.

**Se conserva:** comparación original de cinco motores y reproducción de eventos NASA.

**Se añade al final:** práctica PQRS con Polars CPU/GPU: mismos grupos, muestras o completos, equivalencia, calentamiento, tres repeticiones y dos fronteras de medición.

La ampliación GPU es independiente: puede ejecutarse desde su primera celda sin repetir los eventos NASA. CPU funciona sin NVIDIA; GPU se solicita explícitamente en un entorno compatible. No añade otra entrega de P04.


# P04 e I01. Rendimiento y traza finita

P04: clase 7, 23 de octubre; I01: clase 8, 24 de octubre de 2026.
Preparado por el PhD Esteban Hernández, CyberColombia.

[Enunciado P04](../../talleres/P04.md) · [I01](../../talleres/I01.md) · [WSL](../../docs/instalacion-wsl.md) · [Linux/macOS](../../docs/instalacion-linux-macos.md)

El viernes el profesor demuestra; el sábado los estudiantes ejecutan y entregan en clase. Ejecutar las celdas en orden con el kernel del único `.venv`. Cada lectura, consulta y escritura queda visible. No se importan scripts del curso.

## 1. Preparar y localizar los datos
Desde la terminal en `kit/`, con `.venv` activo:
```bash
python 00_datos.py --descargar pqrs nasa
python 00_datos.py --verificar pqrs nasa
```
Primero ejecutar [02 Parquet y contratos](02_Parquet_y_contratos.ipynb): allí se leen los CSV, se convierten los tipos y se escriben el CSV proyectado, el Parquet y sus particiones. Aquí se leen esos productos; no se regeneran silenciosamente. Las muestras contienen 3.000 reportes; no permiten concluir sobre rendimiento nacional.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

# Editar esta ruta para que coincida con el clon local.
# Linux/macOS: Path.home() / "bigdata/big-data-postgraduate"
repositorio = Path("/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate")
KIT = repositorio / "kit"
RAW = KIT / "data/raw"
OUT = KIT / "salidas" / "pandas/P04"
assert KIT.is_dir(), "Corrige repositorio antes de continuar"
OUT.mkdir(parents=True, exist_ok=True)


In [ ]:
entrada = KIT / "salidas/pandas/P02"
archivo_parquet = entrada / "pqrs.parquet"
archivo_csv = entrada / "pqrs_proyeccion.csv"
particiones = entrada / "pqrs_particionado"
assert archivo_parquet.is_file(), "Ejecuta el notebook 02"
assert archivo_csv.is_file() and list(particiones.glob("**/*.parquet"))
control_entrada = json.loads((entrada / "control_formatos_pandas.json").read_text())
assert control_entrada["filas"] == 3000, "Este bloque mide las muestras de P02"
columnas = ["corte_archivo", "anio", "mes_num", "afec_cod_depto"]
# Importar antes de medir: excluimos el coste de importación de las bibliotecas.
import polars as pl
import duckdb
from time import perf_counter
import platform
mediciones = []
display(pd.DataFrame([{"python": platform.python_version(), "pandas": pd.__version__,
                       "polars": pl.__version__, "duckdb": duckdb.__version__,
                       "sistema": platform.platform()}]))


## 2. Comparar la misma consulta en cinco variantes
Agrupar **todos los años** por corte, año, mes y departamento, incluyendo claves nulas, y contar filas. Cada celda mide lectura + agregación + obtención del resultado en pandas. `%%time` informa CPU y **Wall time**; `perf_counter` guarda el tiempo transcurrido del bloque para el informe. Polars incluye `to_pandas`; DuckDB incluye la apertura/cierre de conexión.

La ordenación, normalización de tipos, comprobación y visualización se realizan después y quedan fuera del intervalo registrado. Ejecutar una vuelta de calentamiento, vaciar `mediciones` con la celda indicada y realizar tres vueltas, alternando el orden de los motores. No volver a ejecutar la configuración entre vueltas. La caché del sistema puede persistir; no llamar a esto una prueba con caché fría. Este ejercicio comparte el proceso del kernel y no mide RSS ni arranque de procesos.


### pandas: lectura inmediata y groupby


In [ ]:
%%time
inicio = perf_counter()
datos_pandas = pd.read_parquet(archivo_parquet, columns=columnas)
resultado_pandas = (datos_pandas.groupby(columnas, dropna=False)
    .size().reset_index(name="reportes"))
segundos = perf_counter() - inicio
mediciones.append({"motor": "pandas", "segundos": segundos})


### Polars: lectura diferida y ejecución explícita
`scan_parquet` prepara el plan; `collect` lo ejecuta. Medir solo la creación de `consulta` no mide el procesamiento.


In [ ]:
%%time
inicio = perf_counter()
consulta = (pl.scan_parquet(archivo_parquet)
    .group_by(columnas).agg(pl.len().alias("reportes")))
resultado_polars = consulta.collect(engine="streaming").to_pandas()
segundos = perf_counter() - inicio
mediciones.append({"motor": "polars", "segundos": segundos})


### DuckDB: SQL sobre Parquet
Las tres variantes siguientes usan exactamente el mismo SQL. El CSV se lee como texto para conservar los ceros de los códigos; por ello incluye un coste distinto de interpretación del formato.


In [ ]:
sql = """
SELECT corte_archivo, anio, mes_num, afec_cod_depto, COUNT(*) AS reportes
FROM reportes
GROUP BY corte_archivo, anio, mes_num, afec_cod_depto
"""


### duckdb


In [ ]:
%%time
inicio = perf_counter()
con = duckdb.connect()
try:
    con.read_parquet(str(archivo_parquet)).create_view("reportes")
    resultado_duckdb = con.sql(sql).df()
finally:
    con.close()
segundos = perf_counter() - inicio
mediciones.append({"motor": "duckdb", "segundos": segundos})


### duckdb_particionado


In [ ]:
%%time
inicio = perf_counter()
con = duckdb.connect()
try:
    con.read_parquet(str(particiones / "**/*.parquet"), hive_partitioning=True).create_view("reportes")
    resultado_duckdb_particionado = con.sql(sql).df()
finally:
    con.close()
segundos = perf_counter() - inicio
mediciones.append({"motor": "duckdb_particionado", "segundos": segundos})


### duckdb_csv


In [ ]:
%%time
inicio = perf_counter()
con = duckdb.connect()
try:
    con.read_csv(str(archivo_csv), header=True, all_varchar=True).create_view("reportes")
    resultado_duckdb_csv = con.sql(sql).df()
finally:
    con.close()
segundos = perf_counter() - inicio
mediciones.append({"motor": "duckdb_csv", "segundos": segundos})


## 3. Verificar antes de comparar tiempos
El CSV devuelve claves numéricas como texto; armonizamos tipos fuera de la medición. No basta con comparar el total: cada grupo debe coincidir. Inspeccionar primero los tipos originales. Repetir esta comprobación después de cada vuelta.


In [ ]:
resultados = {"pandas": resultado_pandas, "polars": resultado_polars,
              "duckdb": resultado_duckdb, "duckdb_particionado": resultado_duckdb_particionado,
              "duckdb_csv": resultado_duckdb_csv}
comparables = {}
for motor, resultado in resultados.items():
    print(motor, resultado.dtypes.to_dict())
    normalizado = resultado.copy()
    for columna in ["anio", "mes_num", "reportes"]:
        normalizado[columna] = pd.to_numeric(normalizado[columna], errors="raise").astype("Int64")
    normalizado["corte_archivo"] = normalizado["corte_archivo"].astype("string")
    normalizado["afec_cod_depto"] = normalizado["afec_cod_depto"].astype("string").str.zfill(2)
    comparables[motor] = normalizado.sort_values(columnas).reset_index(drop=True)
    assert normalizado["reportes"].sum() == 3000
for motor, resultado in comparables.items():
    pd.testing.assert_frame_equal(comparables["pandas"], resultado)
display(comparables["pandas"])
print("Los cinco resultados coinciden grupo por grupo")


### Excluir el calentamiento
Ejecutar la siguiente celda **una vez**, tras verificar la primera vuelta. Después repetir únicamente las cinco celdas medidas y la comprobación anterior tres veces; alternar su orden. No ejecutar de nuevo esta celda al finalizar.


In [ ]:
mediciones = []


### Guardar las tres repeticiones
Ejecutar tras completar las tres vueltas. Este archivo documenta el alcance del kernel; no sustituye las mediciones de RSS en procesos independientes.


In [ ]:
tiempos = pd.DataFrame(mediciones)
assert len(tiempos) == 15, "Faltan las tres vueltas de cinco motores después del calentamiento"
assert tiempos.groupby("motor").size().eq(3).all()
tiempos["repeticion"] = tiempos.groupby("motor").cumcount() + 1
tiempos["alcance"] = "kernel: lectura, agregación y resultado pandas; 3000 filas"
resumen = tiempos.groupby("motor")["segundos"].agg(["median", "min", "max"]).reset_index()
tiempos.to_csv(OUT / "benchmark_notebook.csv", index=False)
resumen.to_csv(OUT / "benchmark_notebook_resumen.csv", index=False)
comparables["pandas"].to_csv(OUT / "conteos_verificados.csv", index=False)
display(resumen)


## 4. Escala real y automatización, después de comprender las consultas
[P04](../../talleres/P04.md) conserva la extensión de cinco variantes en procesos independientes, dos hilos, calentamiento, tres repeticiones y sondeo RSS cada 10 ms. Revisar `trabajador` y `benchmark` en `kit/pqrs_talleres.py`: identificar en el código cada consulta que acabas de ejecutar antes de usar la automatización.

Los tiempos del CLI incluyen arranque y exportación; **no mezclar** esas mediciones con las del kernel. El CLI escribe `benchmark_pqrs.csv` y su resumen; el notebook usa otros nombres para distinguir el alcance. Registrar máquina, versiones e hilos. Las bibliotecas del kernel pueden usar distintos números de hilos: esta comparación describe la configuración local, no aísla el efecto del motor.

El volumen completo se descarga explícitamente según P04; reservar al menos 8 GB de disco más el entorno. Con 8 GB de RAM mantener pandas en muestras. Los completos concilian 2.444.766 reportes. No cambiar la ruta de este bloque de muestras a los completos sin revisar su preparación y memoria.

**Detenerse aquí en la clase 7.** Retomar eventos en la clase 8.

## 5. Leer las observaciones NASA
`json.load` lee el documento; `DataFrame` convierte el diccionario de parámetros en filas. Convertimos las fechas y separamos los faltantes (-999), sin inventar lluvia. La fecha diaria se usa como tiempo de evento por convención didáctica.


In [ ]:
with open(RAW / "nasa.json", encoding="utf-8") as archivo:
    nasa = json.load(archivo)
lluvia = pd.Series(nasa["properties"]["parameter"]["PRECTOTCORR"], name="lluvia_mm")
observaciones = lluvia.rename_axis("event_id").reset_index()
observaciones["fecha_evento"] = pd.to_datetime(observaciones["event_id"], format="%Y%m%d")
observaciones["lluvia_mm"] = pd.to_numeric(observaciones["lluvia_mm"], errors="raise").replace(-999, float("nan"))
assert observaciones["event_id"].is_unique
display(observaciones.head(12))


## 6. Construir el orden de entrega y unir los valores reales
Primero llegan los días 1–4 y 6–12; luego se reintenta el 6 y finalmente llega el 5. `merge(validate="many_to_one")` permite entregas repetidas pero exige una observación por fecha. Inspeccionar la tabla antes de aplicar reglas.


In [ ]:
dias = [1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 6, 5]
entregas = pd.DataFrame({"event_id": [f"202501{dia:02d}" for dia in dias]})
entregas["orden_entrega"] = range(1, len(entregas) + 1)
entregas = entregas.merge(observaciones, on="event_id", how="left", validate="many_to_one", indicator=True)
assert entregas["_merge"].eq("both").all() and entregas["lluvia_mm"].notna().all()
display(entregas)


## 7. Recorrer la traza paso a paso
Antes de cada entrega, el límite es la fecha máxima observada menos tres días. Primero comprobamos si el identificador ya fue aceptado; después, si la fecha es anterior al límite. Una fecha igual al límite se admite. La lista `traza` conserva la decisión y el límite usado; `vistos` solo guarda aceptados. Esta regla finita no reproduce toda la semántica de Spark.


In [ ]:
vistos = set()
ultima_fecha = None
traza = []
for entrega in entregas.itertuples(index=False):
    limite = ultima_fecha - pd.Timedelta(days=3) if ultima_fecha is not None else None
    if entrega.event_id in vistos:
        estado = "duplicado"
    elif limite is not None and entrega.fecha_evento < limite:
        estado = "tardio"
    else:
        estado = "aceptado"
        vistos.add(entrega.event_id)
    traza.append({"orden_entrega": entrega.orden_entrega, "event_id": entrega.event_id,
                  "fecha_evento": entrega.fecha_evento, "lluvia_mm": entrega.lluvia_mm,
                  "watermark_antes": limite, "estado": estado})
    ultima_fecha = max(ultima_fecha, entrega.fecha_evento) if ultima_fecha is not None else entrega.fecha_evento
    print(entrega.orden_entrega, entrega.event_id, "límite:", limite, estado)
traza = pd.DataFrame(traza)


## 8. Conciliar y escribir las evidencias
Comparar entregas, fechas distintas y aceptados. Explicar qué observación falta en la suma aceptada y proponer una corrección antes de continuar con Spark en E11.


In [ ]:
aceptados = traza.loc[traza["estado"].eq("aceptado")]
control = {"entregas": len(traza), "registros_reales_distintos": int(traza["event_id"].nunique()),
           "aceptados": len(aceptados), "duplicados": int(traza["estado"].eq("duplicado").sum()),
           "tardios": int(traza["estado"].eq("tardio").sum()),
           "suma_aceptada_mm": float(aceptados["lluvia_mm"].sum()),
           "suma_fechas_unicas_mm": float(entregas.drop_duplicates("event_id")["lluvia_mm"].sum())}
assert (control["entregas"], control["registros_reales_distintos"], control["aceptados"], control["duplicados"], control["tardios"]) == (13, 12, 11, 1, 1)
assert abs(control["suma_aceptada_mm"] - 41.44) < 1e-8
traza.to_csv(OUT / "traza_eventos_nasa.csv", index=False)
with open(OUT / "control_eventos_intro.json", "w", encoding="utf-8") as archivo:
    json.dump(control, archivo, ensure_ascii=False, indent=2)
recuperada = pd.read_csv(OUT / "traza_eventos_nasa.csv", dtype={"event_id": "string"},
                         parse_dates=["fecha_evento", "watermark_antes"])
pd.testing.assert_frame_equal(traza.astype({"event_id": "string"}), recuperada)
display(control)
display(recuperada)


## Entrega
Adjuntar tiempos, resultados equivalentes, configuración y alcance de la medición; distinguir notebook y CLI. Entregar la traza y explicar el reintento y el atraso. Modificar una condición de consulta, anticipar el resultado y verificarlo en todos los motores. No sumar lluvia con PQRS ni fabricar tiempos de evento para reportes.


# Ampliación update1 · PQRS con Polars CPU y GPU
Curso de Postgrado: BigData, Especialización en Bases de datos · CyberColombia.

Ampliación de [P04](../../talleres/P04.md), sin otra entrega. Ejecutar primero
el notebook 04 para comprender las consultas. Aquí conservamos su agrupación
por corte, año, mes y departamento. No descarga, prepara ni modifica originales.

CPU funciona de forma independiente. Para GPU preparar `.venv-gpu` siguiendo
[la guía](../../docs/polars-gpu.md) y seleccionar su kernel. No instalar desde
este notebook. Polars del curso: 1.35.2; el backend debe ser compatible.
La prueba CUDA queda pendiente hasta ejecutarla en hardware NVIDIA real.


## 1. Elegir alcance y datos
Muestras: completar P02, entrada `kit/salidas/pandas/P02/pqrs.parquet`.
Completos: seguir P04 y ejecutar `formatos --completo`; entrada
`kit/salidas/pqrs_completos/pqrs.parquet`. No basta con descargar CSV.
No apilar copias para simular nuevos reportes. Ajustar `REPOSITORIO` si hace falta.


In [ ]:
from pathlib import Path
from time import perf_counter
from importlib import metadata
import json, platform, sys, statistics
import polars as pl
from polars.testing import assert_frame_equal

REPOSITORIO = None  # Ejemplo: Path('/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate')
ALCANCE = 'muestra'  # 'muestra' o 'completo'
EJECUTAR_GPU = False  # True solo después de verificar el entorno NVIDIA
REPETICIONES = 3
if REPOSITORIO is None:
    REPOSITORIO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                       if (p / 'kit/pqrs_fuentes.json').is_file()), None)
assert REPOSITORIO is not None, 'Configurar REPOSITORIO'
REPOSITORIO = Path(REPOSITORIO).expanduser().resolve()
assert ALCANCE in {'muestra', 'completo'}
assert REPETICIONES >= 3
KIT = REPOSITORIO / 'kit'
ENTRADA = KIT / 'salidas' / ('pandas/P02' if ALCANCE == 'muestra' else 'pqrs_completos')
ARCHIVO = ENTRADA / 'pqrs.parquet'
assert ARCHIVO.is_file(), f'Preparar datos antes: {ARCHIVO}'
OUT = KIT / 'salidas/pandas/P04_GPU' / ALCANCE
OUT.mkdir(parents=True, exist_ok=True)
CLAVES = ['corte_archivo', 'anio', 'mes_num', 'afec_cod_depto']
fuentes = json.loads((KIT / 'pqrs_fuentes.json').read_text())
ESPERADAS = 3000 if ALCANCE == 'muestra' else sum(f['filas'] for f in fuentes)
versiones = {}
for paquete in ['polars', 'cudf-polars-cu12', 'cudf-polars-cu13']:
    try:
        versiones[paquete] = metadata.version(paquete)
    except metadata.PackageNotFoundError:
        versiones[paquete] = None
print(sys.executable, platform.platform(), versiones)
print('Entrada:', ARCHIVO, 'Bytes Parquet:', ARCHIVO.stat().st_size)


## 2. Una consulta, dos motores
`pl.len()` cuenta filas, también cuando alguna clave es nula. Se mantienen
los códigos como cadenas y no se excluyen registros. El total son reportes,
no personas únicas. La ordenación para comparar queda fuera de la medición.


In [ ]:
def agregar(base):
    return base.group_by(CLAVES).agg(pl.len().alias('reportes'))

consulta = agregar(pl.scan_parquet(ARCHIVO).select(CLAVES))
print(consulta.explain())
inicio = perf_counter()
referencia = consulta.collect(engine='in-memory')
primera_cpu_s = perf_counter() - inicio
assert int(referencia['reportes'].sum()) == ESPERADAS
referencia_ordenada = referencia.sort(CLAVES)
print(referencia_ordenada.head(), 'Total:', ESPERADAS)


## 3. Verificar GPU sin fallback silencioso
`raise_on_fail=True` hace visible un plan no soportado. No capturamos ese error
para continuar como si se hubiera ejecutado GPU. `collect` devuelve un DataFrame
en memoria CPU; la medición incluye completar esa devolución.


In [ ]:
engine_gpu = None
primera_gpu_s = None
gpu_verificada = False
if EJECUTAR_GPU:
    engine_gpu = pl.GPUEngine(raise_on_fail=True)
    inicio = perf_counter()
    prueba_gpu = consulta.collect(engine=engine_gpu)
    primera_gpu_s = perf_counter() - inicio
    assert_frame_equal(referencia_ordenada, prueba_gpu.sort(CLAVES), check_exact=True)
    gpu_verificada = True
    del prueba_gpu
else:
    print('GPU NO EJECUTADA. Continuamos únicamente con CPU.')


## 4. Medir lectura, agregación y resultado CPU
La primera ejecución anterior se registra aparte. Hacer tres repeticiones,
alternando el orden CPU/GPU. Se excluyen descarga, preparación, creación del
plan, ordenación, comprobaciones y exportación. No se vacían las cachés del SO.
CPU usa `in-memory`, no el motor streaming del benchmark original: no mezclar
sus tiempos. No se mide RAM pico ni VRAM pico en este notebook.


In [ ]:
mediciones = []
def medir(q, escenario):
    for vuelta in range(REPETICIONES):
        orden = ['cpu', 'gpu'] if vuelta % 2 == 0 else ['gpu', 'cpu']
        for motor in orden:
            if motor == 'gpu' and not gpu_verificada:
                continue
            engine = 'in-memory' if motor == 'cpu' else engine_gpu
            inicio = perf_counter()
            resultado = q.collect(engine=engine)
            segundos = perf_counter() - inicio
            assert_frame_equal(referencia_ordenada, resultado.sort(CLAVES), check_exact=True)
            mediciones.append({'escenario': escenario, 'motor': motor,
                'repeticion': vuelta + 1, 'segundos': segundos,
                'filas_entrada': ESPERADAS, 'grupos': resultado.height,
                'equivalente': True})
            del resultado

medir(consulta, 'parquet_a_resultado_cpu')


## 5. Separar la lectura del archivo
Materializar únicamente las cuatro columnas de entrada en RAM fuera de la
medición. Al ejecutar esta consulta en GPU siguen existiendo transferencias:
**entrada en RAM no significa datos residentes en VRAM**. Este escenario no
aísla el tiempo del kernel GPU ni mide H2D/cálculo/D2H por separado.
Con completos, verificar primero capacidad de RAM y VRAM para entrada y
temporales; el tamaño comprimido de Parquet no es la memoria necesaria.


In [ ]:
inicio = perf_counter()
entrada_ram = pl.read_parquet(ARCHIVO, columns=CLAVES)
lectura_ram_s = perf_counter() - inicio
assert entrada_ram.height == ESPERADAS
bytes_estimados_ram = entrada_ram.estimated_size()
print('Entrada RAM estimada:', bytes_estimados_ram, 'bytes; lectura:', lectura_ram_s)
consulta_ram = agregar(entrada_ram.lazy())
consulta_ram.collect(engine='in-memory')  # Calentamiento no medido
if gpu_verificada:
    calentamiento_gpu = consulta_ram.collect(engine=engine_gpu)
    assert_frame_equal(referencia_ordenada, calentamiento_gpu.sort(CLAVES), check_exact=True)
    del calentamiento_gpu
medir(consulta_ram, 'ram_a_resultado_cpu')


## 6. Resumen e informe
Aceleración = mediana CPU / mediana GPU para el mismo escenario.
Mayor que 1 favorece GPU; menor que 1 favorece CPU. Si no se ejecutó GPU,
el factor es nulo, nunca cero ni una estimación. El CSV contiene solo ejecuciones
completadas y equivalentes. RSS mide host, no VRAM; aquí ambas quedan sin medir.
Los archivos reemplazan únicamente los resultados de esta ampliación.


In [ ]:
tiempos = pl.DataFrame(mediciones)
resumen = tiempos.group_by(['escenario', 'motor']).agg(
    pl.col('segundos').median().alias('mediana_s'),
    pl.col('segundos').min().alias('min_s'),
    pl.col('segundos').max().alias('max_s'),
    pl.len().alias('repeticiones')).sort(['escenario', 'motor'])
factores = {}
for escenario in tiempos['escenario'].unique():
    filas = resumen.filter(pl.col('escenario') == escenario)
    por_motor = dict(zip(filas['motor'].to_list(), filas['mediana_s'].to_list()))
    factores[escenario] = (por_motor['cpu'] / por_motor['gpu']
                          if 'gpu' in por_motor else None)
control = {'alcance': ALCANCE, 'entrada': str(ARCHIVO),
    'filas': ESPERADAS, 'claves': CLAVES, 'gpu_solicitada': EJECUTAR_GPU,
    'gpu_ejecutada_y_equivalente': gpu_verificada,
    'primera_cpu_s': primera_cpu_s, 'primera_gpu_s': primera_gpu_s,
    'lectura_ram_s': lectura_ram_s, 'entrada_ram_estimada_bytes': bytes_estimados_ram,
    'repeticiones': REPETICIONES, 'hilos_polars': pl.thread_pool_size(),
    'python': sys.version, 'ejecutable': sys.executable,
    'plataforma': platform.platform(), 'versiones': versiones,
    'aceleracion_cpu_sobre_gpu': factores,
    'ram_pico_bytes': None, 'vram_pico_bytes': None,
    'descarga_y_preparacion_incluidas': False,
    'interpretacion_docente': 'PENDIENTE: completar las preguntas de cierre'}
tiempos.write_csv(OUT / 'benchmark_polars_cpu_gpu.csv')
resumen.write_csv(OUT / 'resumen_polars_cpu_gpu.csv')
referencia_ordenada.write_csv(OUT / 'conteos_verificados.csv')
(OUT / 'control_polars_gpu.json').write_text(json.dumps(control, ensure_ascii=False, indent=2))
print(resumen)
print('Factores:', factores, 'Salidas:', OUT)


## 7. Interpretar antes de entregar
1. ¿Coinciden todos los grupos y el total? ¿Cuántos reportes y cuántos grupos hay?
2. ¿Se ejecutó GPU realmente? Adjuntar modelo, VRAM, controlador y versiones
   con el registro de entorno de la guía GPU; no inventar valores si no está disponible.
3. ¿Qué cambia entre lectura Parquet y entrada RAM? ¿Qué transferencias permanecen?
4. ¿Hay mejora en la mediana? Mostrar también primera ejecución y dispersión.
5. ¿Por qué una muestra de 3.000 filas no permite concluir sobre escala nacional?
6. ¿Qué pasaría con más grupos o una clave de alta cardinalidad?
7. ¿Qué mejorarías antes de usar un clúster Spark?

Completar la interpretación en el informe único de P04 junto a sus CSV/JSON.
No mezclar estos tiempos con el CLI de cinco motores: las fronteras e hilos
son diferentes. La GPU no acelera automáticamente la descarga de Supersalud.

Fuentes: [Polars GPU](https://docs.pola.rs/user-guide/gpu-support/),
[preparación del curso](../../docs/polars-gpu.md).
